In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00004-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00005-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00006-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00007-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00008-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00009-363d1b

In [2]:

# ============================================================
# O1 - STEP 1: DATASET INVENTORY AND STRUCTURAL INSPECTION
# Thesis: Cross-Dataset Multiclass IoT Intrusion Detection
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DATASET PATHS
# ------------------------------------------------------------

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

IOT23_PATH = "/kaggle/input/datasets/akashdogra/cic-iot-2023"

OUTPUT_PATH = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ------------------------------------------------------------
# 2. VERIFY DATASET DIRECTORIES
# ------------------------------------------------------------

datasets = {
    "CIC_IoT_DIAD_2024": DIAD_PATH,
    "CIC_IoT_2023": IOT23_PATH
}

for name, path in datasets.items():
    print("\n" + "=" * 65)
    print("DATASET:", name)
    print("=" * 65)

    if os.path.exists(path):
        print("Status: AVAILABLE")
    else:
        print("Status: NOT FOUND")
        continue

    files = glob.glob(
        os.path.join(path, "**", "*.csv"),
        recursive=True
    )

    print("Total CSV files:", len(files))

# ------------------------------------------------------------
# 3. INSPECT ALL CSV FILES
# ------------------------------------------------------------

inventory = []

for dataset_name, dataset_path in datasets.items():

    csv_files = glob.glob(
        os.path.join(dataset_path, "**", "*.csv"),
        recursive=True
    )

    for file_path in csv_files:

        file_size_mb = os.path.getsize(file_path) / (1024**2)

        record = {
            "Dataset": dataset_name,
            "File": os.path.basename(file_path),
            "File_Path": file_path,
            "Size_MB": round(file_size_mb, 3),
            "Columns_Count": None,
            "Columns": None,
            "Read_Status": None,
            "Possible_Label_Columns": None,
            "Possible_Time_Columns": None,
            "Possible_ID_Columns": None
        }

        try:
            # Read only five records for structural inspection
            sample = pd.read_csv(
                file_path,
                nrows=5,
                low_memory=False
            )

            columns = sample.columns.tolist()

            record["Columns_Count"] = len(columns)
            record["Columns"] = " | ".join(map(str, columns))
            record["Read_Status"] = "OK"

            # Identify possible label columns
            record["Possible_Label_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["label", "class", "attack", "category"])
            )

            # Identify possible timestamp columns
            record["Possible_Time_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["time", "date", "timestamp"])
            )

            # Identify possible device/group identifiers
            record["Possible_ID_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["device", "source", "capture", "group"])
            )

        except Exception as e:
            record["Read_Status"] = str(e)[:200]

        inventory.append(record)

# ------------------------------------------------------------
# 4. CREATE INVENTORY TABLE
# ------------------------------------------------------------

inventory_df = pd.DataFrame(inventory)

print("\n" + "=" * 65)
print("DATASET INVENTORY SUMMARY")
print("=" * 65)

print(
    inventory_df.groupby("Dataset")
    .agg(
        Total_Files=("File", "count"),
        Total_Size_MB=("Size_MB", "sum"),
        Successful_Reads=(
            "Read_Status",
            lambda x: (x == "OK").sum()
        )
    )
)

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

inventory_file = os.path.join(
    OUTPUT_PATH,
    "O1_dataset_inventory.csv"
)

inventory_df.to_csv(inventory_file, index=False)

print("\nInventory saved successfully:")
print(inventory_file)

print("\nFirst 10 inventory records:")
display(inventory_df.head(10))

print("\nO1 STEP 1 COMPLETED")



DATASET: CIC_IoT_DIAD_2024
Status: AVAILABLE
Total CSV files: 126

DATASET: CIC_IoT_2023
Status: AVAILABLE
Total CSV files: 169

DATASET INVENTORY SUMMARY
                   Total_Files  Total_Size_MB  Successful_Reads
Dataset                                                        
CIC_IoT_2023               169      13116.991               169
CIC_IoT_DIAD_2024          126       9819.528               126

Inventory saved successfully:
/kaggle/working/O1_results/O1_dataset_inventory.csv

First 10 inventory records:


,Dataset,File,File_Path,Size_MB,Columns_Count,Columns,Read_Status,Possible_Label_Columns,Possible_Time_Columns,Possible_ID_Columns
0,CIC_IoT_DIAD_2024,BenignTraffic.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,94.622,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
1,CIC_IoT_DIAD_2024,BenignTraffic1.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,44.293,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
2,CIC_IoT_DIAD_2024,BenignTraffic2.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,47.887,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
3,CIC_IoT_DIAD_2024,BenignTraffic3.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,20.406,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
4,CIC_IoT_DIAD_2024,DictionaryBruteForce.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,1.666,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
5,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,152.219,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
6,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation1.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,67.653,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
7,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation10.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,81.452,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
8,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation11.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,97.194,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
9,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation12.pcap_Flow.csv,/kaggle/input/datasets/aymeneinformatique/cic-...,18.438,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,



O1 STEP 1 COMPLETED


In [3]:

# ============================================================
# O1 - STEP 2: DATA QUALITY ASSESSMENT
# Missing values, infinities, duplicates and data types
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CHUNK_SIZE = 100_000

datasets = {
    "CIC_IoT_DIAD_2024": DIAD_PATH,
    "CIC_IoT_2023": IOT23_PATH
}

quality_results = []
column_results = []

# ------------------------------------------------------------
# 2. PROCESS EACH CSV FILE IN CHUNKS
# ------------------------------------------------------------

for dataset_name, dataset_path in datasets.items():

    print("\n" + "=" * 70)
    print("DATA QUALITY:", dataset_name)
    print("=" * 70)

    files = glob.glob(
        os.path.join(dataset_path, "**", "*.csv"),
        recursive=True
    )

    for file_number, file_path in enumerate(files, 1):

        print(
            f"\rProcessing {file_number}/{len(files)}: "
            f"{os.path.basename(file_path)[:45]}",
            end=""
        )

        row_count = 0
        missing_counts = defaultdict(int)
        infinity_counts = defaultdict(int)
        dtype_observations = defaultdict(set)
        duplicate_count = 0
        column_names = []
        read_error = None

        try:
            reader = pd.read_csv(
                file_path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

            for chunk in reader:

                row_count += len(chunk)

                if not column_names:
                    column_names = chunk.columns.tolist()

                # Missing values
                missing = chunk.isna().sum()

                for col, count in missing.items():
                    missing_counts[col] += int(count)

                # Numerical data types and infinities
                numeric = chunk.select_dtypes(
                    include=[np.number]
                )

                if not numeric.empty:

                    inf_counts = np.isinf(
                        numeric.to_numpy()
                    ).sum(axis=0)

                    for col, count in zip(
                        numeric.columns, inf_counts
                    ):
                        infinity_counts[col] += int(count)

                # Record observed data types
                for col in chunk.columns:
                    dtype_observations[col].add(
                        str(chunk[col].dtype)
                    )

                # Exact duplicate rows within each chunk
                duplicate_count += int(
                    chunk.duplicated().sum()
                )

            # File-level summary
            quality_results.append({
                "Dataset": dataset_name,
                "File": os.path.basename(file_path),
                "Rows": row_count,
                "Columns": len(column_names),
                "Missing_Values": sum(missing_counts.values()),
                "Infinite_Values": sum(infinity_counts.values()),
                "Within_Chunk_Duplicates": duplicate_count,
                "Read_Status": "OK"
            })

            # Column-level summary
            for col in column_names:

                column_results.append({
                    "Dataset": dataset_name,
                    "File": os.path.basename(file_path),
                    "Column": col,
                    "Missing_Count": missing_counts[col],
                    "Missing_Percent": (
                        missing_counts[col] / row_count * 100
                        if row_count else 0
                    ),
                    "Infinite_Count": infinity_counts[col],
                    "Observed_Dtypes": ", ".join(
                        sorted(dtype_observations[col])
                    )
                })

        except Exception as e:

            read_error = str(e)[:300]

            quality_results.append({
                "Dataset": dataset_name,
                "File": os.path.basename(file_path),
                "Rows": row_count,
                "Columns": len(column_names),
                "Missing_Values": None,
                "Infinite_Values": None,
                "Within_Chunk_Duplicates": None,
                "Read_Status": read_error
            })

    print("\nFinished:", dataset_name)

# ------------------------------------------------------------
# 3. CREATE RESULTS TABLES
# ------------------------------------------------------------

quality_df = pd.DataFrame(quality_results)
columns_quality_df = pd.DataFrame(column_results)

print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

summary = quality_df.groupby("Dataset").agg(
    Total_Rows=("Rows", "sum"),
    Total_Files=("File", "count"),
    Total_Missing=("Missing_Values", "sum"),
    Total_Infinite=("Infinite_Values", "sum"),
    Within_Chunk_Duplicates=(
        "Within_Chunk_Duplicates", "sum"
    )
)

display(summary)

# ------------------------------------------------------------
# 4. TOP MISSING-VALUE FEATURES
# ------------------------------------------------------------

print("\nTOP 20 COLUMNS WITH MISSING VALUES")

missing_summary = (
    columns_quality_df.groupby(
        ["Dataset", "Column"]
    )["Missing_Count"]
    .sum()
    .reset_index()
    .sort_values(
        "Missing_Count",
        ascending=False
    )
)

display(missing_summary.head(20))

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

quality_df.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_data_quality_summary.csv"
    ),
    index=False
)

columns_quality_df.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_column_quality_details.csv"
    ),
    index=False
)

missing_summary.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_missing_value_summary.csv"
    ),
    index=False
)

print("\nAll quality reports saved in:", OUTPUT_PATH)
print("O1 STEP 2 COMPLETED")



DATA QUALITY: CIC_IoT_DIAD_2024
Processing 126/126: MITM-ArpSpoofing1.pcap_Flow.csv
Finished: CIC_IoT_DIAD_2024

DATA QUALITY: CIC_IoT_2023
Processing 169/169: part-00168-363d1ba3-8ab5-4f96-bc25-4d5862db7c
Finished: CIC_IoT_2023

DATA QUALITY SUMMARY


,Total_Rows,Total_Files,Total_Missing,Total_Infinite,Within_Chunk_Duplicates
Dataset,,,,,
CIC_IoT_2023,46686579,169,0,0,0
CIC_IoT_DIAD_2024,19507834,126,85507,91589,32930



TOP 20 COLUMNS WITH MISSING VALUES


,Dataset,Column,Missing_Count
281,CIC_IoT_DIAD_2024,Flow Bytes/s,85507
228,CIC_IoT_DIAD_2024,58.68986283848344,0
227,CIC_IoT_DIAD_2024,57126.0.2,0
226,CIC_IoT_DIAD_2024,57126.0.1,0
225,CIC_IoT_DIAD_2024,57126.0,0
224,CIC_IoT_DIAD_2024,57126,0
223,CIC_IoT_DIAD_2024,55338,0
222,CIC_IoT_DIAD_2024,532845.5,0
221,CIC_IoT_DIAD_2024,51.1,0
209,CIC_IoT_DIAD_2024,416190.5.1,0



All quality reports saved in: /kaggle/working/O1_results
O1 STEP 2 COMPLETED


In [4]:
# ============================================================
# O1 STEP 3: DIAD CSV HEADER CONSISTENCY AUDIT
# ============================================================

import os
import glob
import csv
import re
import pandas as pd
from collections import Counter, defaultdict

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

files = glob.glob(os.path.join(DIAD_PATH, "**", "*.csv"), recursive=True)

# Common expected traffic-feature/header terms
known_terms = [
    "label", "timestamp", "flow id", "src ip", "dst ip",
    "src port", "dst port", "protocol", "flow duration",
    "flow bytes", "flow packets", "packet length",
    "fwd", "bwd", "total length", "header length",
    "flow packets/s", "flow bytes/s"
]

def read_first_line(path):
    """Read the first physical line of a CSV file."""
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            return f.readline().strip()
    except Exception as e:
        return f"READ_ERROR: {e}"

def assess_header(path):
    first_line = read_first_line(path)

    try:
        columns = next(csv.reader([first_line]))
    except Exception:
        columns = []

    normalized = [str(c).strip().lower() for c in columns]
    joined = " ".join(normalized)

    matched_terms = [term for term in known_terms if term in joined]

    # A header is more likely when recognizable feature names are present.
    # This is a screening rule, not a definitive proof.
    likely_header = len(matched_terms) >= 2

    return {
        "File": os.path.basename(path),
        "Column_Count": len(columns),
        "Likely_Header": likely_header,
        "Recognized_Header_Terms": ", ".join(matched_terms),
        "First_Line_Sample": first_line[:250]
    }

audit_results = [assess_header(path) for path in files]
header_audit = pd.DataFrame(audit_results)

print("TOTAL DIAD FILES:", len(header_audit))
print("\nHEADER CLASSIFICATION:")
print(header_audit["Likely_Header"].value_counts(dropna=False))

print("\nFILES THAT MAY HAVE NO HEADER:")
display(
    header_audit[header_audit["Likely_Header"] == False][
        ["File", "Column_Count", "First_Line_Sample"]
    ].head(30)
)

print("\nFILES WITH LIKELY HEADERS:")
display(
    header_audit[header_audit["Likely_Header"] == True][
        ["File", "Column_Count", "Recognized_Header_Terms"]
    ].head(15)
)

# Save audit
audit_path = "/kaggle/working/O1_results/O1_DIAD_header_audit.csv"
header_audit.to_csv(audit_path, index=False)

print("\nHeader audit saved to:", audit_path)

TOTAL DIAD FILES: 126

HEADER CLASSIFICATION:
Likely_Header
True     121
False      5
Name: count, dtype: int64

FILES THAT MAY HAVE NO HEADER:


,File,Column_Count,First_Line_Sample
73,DoS-TCP_Flood.pcap_Flow.csv,84,"35.185.101.66-192.168.137.72-443-33777-6,35.18..."
74,DoS-TCP_Flood1.pcap_Flow.csv,84,"192.168.137.164-192.168.137.39-80-34050-6,192...."
75,DoS-TCP_Flood10.pcap_Flow.csv,84,"44.231.176.168-192.168.137.19-443-47717-6,44.2..."
76,DoS-TCP_Flood2.pcap_Flow.csv,84,"192.168.137.31-172.253.115.95-50439-443-6,192...."
77,DoS-TCP_Flood3.pcap_Flow.csv,84,"142.251.41.67-192.168.137.93-80-55338-6,142.25..."



FILES WITH LIKELY HEADERS:


,File,Column_Count,Recognized_Header_Terms
0,BenignTraffic.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
1,BenignTraffic1.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
2,BenignTraffic2.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
3,BenignTraffic3.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
4,DictionaryBruteForce.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
7,DDoS-ACK_Fragmentation10.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
8,DDoS-ACK_Fragmentation11.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
9,DDoS-ACK_Fragmentation12.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."



Header audit saved to: /kaggle/working/O1_results/O1_DIAD_header_audit.csv


In [5]:
# ============================================================
# O1 STEP 3A: FIX FILE PATHS AND VERIFY DIAD SCHEMA
# ============================================================

import os
import glob
import csv
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"
OUTPUT_PATH = "/kaggle/working/O1_results"

# Find actual CSV paths recursively
all_files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

# Map filenames to actual paths
file_path_map = {
    os.path.basename(path): path
    for path in all_files
}

audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = audit.loc[
    audit["Likely_Header"] == False, "File"
].tolist()

headered_files = audit.loc[
    audit["Likely_Header"] == True, "File"
].tolist()

print("Actual CSV files found:", len(all_files))

# Find a reference file that exists
reference_name = next(
    (name for name in headered_files if name in file_path_map),
    None
)

if reference_name is None:
    raise FileNotFoundError("No reference headered file was found.")

reference_path = file_path_map[reference_name]

reference_columns = pd.read_csv(
    reference_path,
    nrows=0
).columns.tolist()

print("\nREFERENCE FILE:", reference_name)
print("REFERENCE COLUMN COUNT:", len(reference_columns))

# Check all headered files
schema_mismatches = []

for filename in headered_files:
    if filename not in file_path_map:
        schema_mismatches.append(filename)
        continue

    columns = pd.read_csv(
        file_path_map[filename],
        nrows=0
    ).columns.tolist()

    if columns != reference_columns:
        schema_mismatches.append(filename)

print("\nHEADERED FILES:", len(headered_files))
print("SCHEMA MISMATCHES:", len(schema_mismatches))
print("MISMATCHED FILES:", schema_mismatches)

# Check headerless files
results = []

for filename in headerless_files:
    if filename not in file_path_map:
        results.append({
            "File": filename,
            "Field_Count": None,
            "Matches_Reference": False,
            "Status": "FILE NOT FOUND"
        })
        continue

    with open(
        file_path_map[filename],
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:
        first_line = f.readline()

    field_count = len(next(csv.reader([first_line])))

    results.append({
        "File": filename,
        "Field_Count": field_count,
        "Matches_Reference": field_count == len(reference_columns),
        "Status": "OK" if field_count == len(reference_columns)
                  else "FIELD COUNT MISMATCH"
    })

print("\nHEADERLESS FILE VERIFICATION:")
display(pd.DataFrame(results))

print("\nREFERENCE COLUMN NAMES:")
print(reference_columns)

Actual CSV files found: 126

REFERENCE FILE: BenignTraffic.pcap_Flow.csv
REFERENCE COLUMN COUNT: 84

HEADERED FILES: 121
SCHEMA MISMATCHES: 0
MISMATCHED FILES: []

HEADERLESS FILE VERIFICATION:


,File,Field_Count,Matches_Reference,Status
0,DoS-TCP_Flood.pcap_Flow.csv,84,True,OK
1,DoS-TCP_Flood1.pcap_Flow.csv,84,True,OK
2,DoS-TCP_Flood10.pcap_Flow.csv,84,True,OK
3,DoS-TCP_Flood2.pcap_Flow.csv,84,True,OK
4,DoS-TCP_Flood3.pcap_Flow.csv,84,True,OK



REFERENCE COLUMN NAMES:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag 

In [6]:
# ============================================================
# O1 STEP 3B: CORRECTED DIAD DATA-QUALITY SCAN
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"
OUTPUT_PATH = "/kaggle/working/O1_results"
CHUNK_SIZE = 100_000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Locate actual files
files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in files}

# Load verified header audit
audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = set(
    audit.loc[audit["Likely_Header"] == False, "File"]
)

# Obtain the verified 84-column schema
reference_file = next(
    name for name in audit.loc[
        audit["Likely_Header"] == True, "File"
    ]
    if name in file_map
)

reference_columns = pd.read_csv(
    file_map[reference_file], nrows=0
).columns.tolist()

print("Verified schema columns:", len(reference_columns))
print("Headerless files:", len(headerless_files))

total_rows = 0
total_missing = 0
total_infinite = 0
total_duplicates = 0
file_results = []
column_missing = {}
read_errors = []

for file_number, (filename, path) in enumerate(file_map.items(), 1):

    print(f"Processing {file_number}/{len(file_map)}: {filename}")

    file_rows = 0
    file_missing = 0
    file_infinite = 0
    file_duplicates = 0

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                path,
                header=None,
                names=reference_columns,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            file_rows += len(chunk)

            # Missing values
            missing_counts = chunk.isna().sum()

            file_missing += int(missing_counts.sum())

            for col, count in missing_counts.items():
                column_missing[col] = (
                    column_missing.get(col, 0) + int(count)
                )

            # Infinite values in numeric columns
            numeric = chunk.select_dtypes(include=[np.number])

            if not numeric.empty:
                values = numeric.to_numpy(dtype=float)
                infinite_mask = np.isinf(values)

                file_infinite += int(infinite_mask.sum())

            # Duplicates within each chunk only
            file_duplicates += int(
                chunk.duplicated().sum()
            )

        total_rows += file_rows
        total_missing += file_missing
        total_infinite += file_infinite
        total_duplicates += file_duplicates

        file_results.append({
            "File": filename,
            "Rows": file_rows,
            "Missing": file_missing,
            "Infinite": file_infinite,
            "Within_Chunk_Duplicates": file_duplicates,
            "Status": "Success"
        })

    except Exception as e:
        read_errors.append({
            "File": filename,
            "Error": str(e)
        })

        file_results.append({
            "File": filename,
            "Rows": file_rows,
            "Missing": file_missing,
            "Infinite": file_infinite,
            "Within_Chunk_Duplicates": file_duplicates,
            "Status": f"Error: {str(e)[:150]}"
        })

# Summary
summary = pd.DataFrame([{
    "Dataset": "CIC_IoT_DIAD_2024",
    "Total_Rows": total_rows,
    "Total_Files": len(files),
    "Total_Missing": total_missing,
    "Total_Infinite": total_infinite,
    "Within_Chunk_Duplicates": total_duplicates,
    "Read_Errors": len(read_errors)
}])

file_report = pd.DataFrame(file_results)

missing_report = pd.DataFrame([
    {"Column": col, "Missing_Count": count}
    for col, count in column_missing.items()
    if count > 0
]).sort_values("Missing_Count", ascending=False)

# Save reports
summary.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_quality_summary.csv"),
    index=False
)

file_report.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_file_quality.csv"),
    index=False
)

missing_report.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_missing_values.csv"),
    index=False
)

print("\n========== CORRECTED DIAD QUALITY SUMMARY ==========")
display(summary)

print("\n========== MISSING VALUE COLUMNS ==========")
display(missing_report)

print("\n========== FILE PROCESSING STATUS ==========")
print(file_report["Status"].value_counts())

if read_errors:
    print("\n========== READ ERRORS ==========")
    display(pd.DataFrame(read_errors))

print("\nReports saved in:", OUTPUT_PATH)

Verified schema columns: 84
Headerless files: 5
Processing 1/126: BenignTraffic.pcap_Flow.csv
Processing 2/126: BenignTraffic1.pcap_Flow.csv
Processing 3/126: BenignTraffic2.pcap_Flow.csv
Processing 4/126: BenignTraffic3.pcap_Flow.csv
Processing 5/126: DictionaryBruteForce.pcap_Flow.csv
Processing 6/126: DDoS-ACK_Fragmentation.pcap_Flow.csv
Processing 7/126: DDoS-ACK_Fragmentation1.pcap_Flow.csv
Processing 8/126: DDoS-ACK_Fragmentation10.pcap_Flow.csv
Processing 9/126: DDoS-ACK_Fragmentation11.pcap_Flow.csv
Processing 10/126: DDoS-ACK_Fragmentation12.pcap_Flow.csv
Processing 11/126: DDoS-ACK_Fragmentation2.pcap_Flow.csv
Processing 12/126: DDoS-ACK_Fragmentation3.pcap_Flow.csv
Processing 13/126: DDoS-ACK_Fragmentation4.pcap_Flow.csv
Processing 14/126: DDoS-ACK_Fragmentation5.pcap_Flow.csv
Processing 15/126: DDoS-ACK_Fragmentation6.pcap_Flow.csv
Processing 16/126: DDoS-ACK_Fragmentation7.pcap_Flow.csv
Processing 17/126: DDoS-ACK_Fragmentation8.pcap_Flow.csv
Processing 18/126: DDoS-ACK_Fr

,Dataset,Total_Rows,Total_Files,Total_Missing,Total_Infinite,Within_Chunk_Duplicates,Read_Errors
0,CIC_IoT_DIAD_2024,19507839,126,85507,91589,32930,0



========== MISSING VALUE COLUMNS ==========


,Column,Missing_Count
0,Flow Bytes/s,85507



========== FILE PROCESSING STATUS ==========
Status
Success    126
Name: count, dtype: int64

Reports saved in: /kaggle/working/O1_results


In [7]:
# ============================================================
# O1 STEP 4A: DIAD CLASS DISTRIBUTION AUDIT
# ============================================================

import os
import glob
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"
OUTPUT_PATH = "/kaggle/working/O1_results"
CHUNK_SIZE = 100_000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Locate all DIAD CSV files
files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in files}

# Load verified header audit
audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = set(
    audit.loc[audit["Likely_Header"] == False, "File"]
)

# Get the verified column names
reference_file = next(
    name for name in audit.loc[
        audit["Likely_Header"] == True, "File"
    ]
    if name in file_map
)

reference_columns = pd.read_csv(
    file_map[reference_file],
    nrows=0
).columns.tolist()

if "Label" not in reference_columns:
    raise ValueError("Label column not found in DIAD reference schema.")

# Count actual labels across all files
overall_counts = {}
file_label_counts = []
total_records = 0
missing_labels = 0
read_errors = []

for number, (filename, path) in enumerate(file_map.items(), 1):

    print(f"Scanning {number}/{len(file_map)}: {filename}")

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                path,
                header=None,
                names=reference_columns,
                usecols=["Label"],
                chunksize=CHUNK_SIZE
            )
        else:
            reader = pd.read_csv(
                path,
                usecols=["Label"],
                chunksize=CHUNK_SIZE
            )

        current_file_counts = {}

        for chunk in reader:
            labels = chunk["Label"]

            missing_labels += int(labels.isna().sum())
            total_records += len(labels)

            # Preserve actual label text, trimming surrounding spaces
            labels = labels.astype("string").str.strip()
            counts = labels.value_counts(dropna=False)

            for label, count in counts.items():
                label_name = (
                    "<MISSING>" if pd.isna(label) else str(label)
                )

                overall_counts[label_name] = (
                    overall_counts.get(label_name, 0) + int(count)
                )

                current_file_counts[label_name] = (
                    current_file_counts.get(label_name, 0) + int(count)
                )

        for label_name, count in current_file_counts.items():
            file_label_counts.append({
                "File": filename,
                "Label": label_name,
                "Count": count
            })

    except Exception as e:
        read_errors.append({
            "File": filename,
            "Error": str(e)
        })

# Overall class distribution
class_distribution = pd.DataFrame([
    {"Label": label, "Count": count}
    for label, count in overall_counts.items()
])

class_distribution = class_distribution.sort_values(
    "Count", ascending=False
).reset_index(drop=True)

class_distribution["Percentage"] = (
    class_distribution["Count"] / total_records * 100
)

# Per-file label distribution
file_distribution = pd.DataFrame(file_label_counts)

# Imbalance ratio, excluding missing labels
valid_counts = [
    count for label, count in overall_counts.items()
    if label != "<MISSING>" and count > 0
]

if valid_counts:
    imbalance_ratio = max(valid_counts) / min(valid_counts)
else:
    imbalance_ratio = None

# Save reports
class_distribution.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_class_distribution.csv"),
    index=False
)

file_distribution.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_file_label_distribution.csv"),
    index=False
)

# Display results
print("\n========== DIAD CLASS DISTRIBUTION ==========")
display(class_distribution)

print("\nTotal records scanned:", total_records)
print("Missing labels:", missing_labels)
print("Number of observed label categories:", len(overall_counts))
print("Class imbalance ratio (largest/smallest non-missing class):",
      imbalance_ratio)

print("\nRead errors:", len(read_errors))

if read_errors:
    display(pd.DataFrame(read_errors))

print("\nReports saved in:", OUTPUT_PATH)

Scanning 1/126: BenignTraffic.pcap_Flow.csv
Scanning 2/126: BenignTraffic1.pcap_Flow.csv
Scanning 3/126: BenignTraffic2.pcap_Flow.csv
Scanning 4/126: BenignTraffic3.pcap_Flow.csv
Scanning 5/126: DictionaryBruteForce.pcap_Flow.csv
Scanning 6/126: DDoS-ACK_Fragmentation.pcap_Flow.csv
Scanning 7/126: DDoS-ACK_Fragmentation1.pcap_Flow.csv
Scanning 8/126: DDoS-ACK_Fragmentation10.pcap_Flow.csv
Scanning 9/126: DDoS-ACK_Fragmentation11.pcap_Flow.csv
Scanning 10/126: DDoS-ACK_Fragmentation12.pcap_Flow.csv
Scanning 11/126: DDoS-ACK_Fragmentation2.pcap_Flow.csv
Scanning 12/126: DDoS-ACK_Fragmentation3.pcap_Flow.csv
Scanning 13/126: DDoS-ACK_Fragmentation4.pcap_Flow.csv
Scanning 14/126: DDoS-ACK_Fragmentation5.pcap_Flow.csv
Scanning 15/126: DDoS-ACK_Fragmentation6.pcap_Flow.csv
Scanning 16/126: DDoS-ACK_Fragmentation7.pcap_Flow.csv
Scanning 17/126: DDoS-ACK_Fragmentation8.pcap_Flow.csv
Scanning 18/126: DDoS-ACK_Fragmentation9.pcap_Flow.csv
Scanning 19/126: DDoS-HTTP_Flood-.pcap_Flow.csv
Scanning 

,Label,Count,Percentage
0,NeedManualLabel,19507839,100.0



Total records scanned: 19507839
Missing labels: 0
Number of observed label categories: 1
Class imbalance ratio (largest/smallest non-missing class): 1.0

Read errors: 0

Reports saved in: /kaggle/working/O1_results


In [8]:
# ============================================================
# O1 STEP 4B: INVESTIGATE DIAD LABEL SOURCE
# ============================================================

import os
import glob
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

# 1. Find README and documentation files
documentation_files = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().startswith("readme") or filename.lower().endswith(
            (".txt", ".md")
        ):
            documentation_files.append(os.path.join(root, filename))

print("DOCUMENTATION FILES FOUND:")
for path in documentation_files:
    print(path)

# 2. Display the first 10 lines of each README/text file
for path in documentation_files:
    print("\nDOCUMENT:", path)

    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for line_number, line in enumerate(f):
                if line_number >= 10:
                    break
                print(line.rstrip())
    except Exception as e:
        print("Could not read:", e)

# 3. Inspect labels in representative files
sample_files = [
    "BenignTraffic.pcap_Flow\\.csv",
    "DictionaryBruteForce.pcap_Flow\\.csv",
    "DDoS-ICMP_Flood.pcap_Flow\\.csv",
    "Mirai-greeth_flood.pcap_Flow\\.csv",
    "MITM-ArpSpoofing.pcap_Flow\\.csv"
]

all_csv = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in all_csv}

print("\nREPRESENTATIVE FILE LABEL INSPECTION:")

for filename in sample_files:
    path = file_map.get(filename)

    if path is None:
        print("\nFILE NOT FOUND:", filename)
        continue

    df = pd.read_csv(path, usecols=["Label"], nrows=20)

    print("\nFILE:", filename)
    print("LABEL VALUES:")
    print(df["Label"].value_counts(dropna=False).to_dict())
    print("SAMPLE LABELS:")
    print(df["Label"].head(5).tolist())

DOCUMENTATION FILES FOUND:

REPRESENTATIVE FILE LABEL INSPECTION:

FILE NOT FOUND: BenignTraffic.pcap_Flow\.csv

FILE NOT FOUND: DictionaryBruteForce.pcap_Flow\.csv

FILE NOT FOUND: DDoS-ICMP_Flood.pcap_Flow\.csv

FILE NOT FOUND: Mirai-greeth_flood.pcap_Flow\.csv

FILE NOT FOUND: MITM-ArpSpoofing.pcap_Flow\.csv


In [9]:
import os

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

print("ACTUAL DIAD CSV FILENAMES")
print("=" * 80)

csv_files = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            full_path = os.path.join(root, filename)
            csv_files.append(full_path)

print("Total CSV files:", len(csv_files))

keywords = [
    "BenignTraffic",
    "DictionaryBruteForce",
    "DDoS-ICMP_Flood",
    "Mirai-greeth_flood",
    "MITM-ArpSpoofing"
]

for keyword in keywords:
    print(f"\nSearching for: {keyword}")

    matches = [
        path for path in csv_files
        if keyword.lower() in os.path.basename(path).lower()
    ]

    if matches:
        for path in matches[:5]:
            print("Filename:", repr(os.path.basename(path)))
            print("Full path:", path)
    else:
        print("No matching filename found.")

print("\nFIRST 15 ACTUAL CSV FILENAMES")
for path in csv_files[:15]:
    print(repr(os.path.basename(path)))

ACTUAL DIAD CSV FILENAMES
Total CSV files: 126

Searching for: BenignTraffic
Filename: 'BenignTraffic.pcap_Flow.csv'
Full path: /kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024/IoT device identification and anomaly detection dataset (CIC IoT-DIAD 2024)/Benign/BenignTraffic.pcap_Flow.csv
Filename: 'BenignTraffic1.pcap_Flow.csv'
Full path: /kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024/IoT device identification and anomaly detection dataset (CIC IoT-DIAD 2024)/Benign/BenignTraffic1.pcap_Flow.csv
Filename: 'BenignTraffic2.pcap_Flow.csv'
Full path: /kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024/IoT device identification and anomaly detection dataset (CIC IoT-DIAD 2024)/Benign/BenignTraffic2.pcap_Flow.csv
Filename: 'BenignTraffic3.pcap_Flow.csv'
Full path: /kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024/IoT device identification and anomaly detection dataset (CIC IoT-DIAD 2024)/Benign/BenignTraffic3.pcap_Flow.csv

Searching for: DictionaryBru

In [10]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

sample_names = [
    "BenignTraffic.pcap_Flow.csv",
    "DictionaryBruteForce.pcap_Flow.csv",
    "DDoS-ICMP_Flood.pcap_Flow.csv",
    "Mirai-greeth_flood.pcap_Flow.csv",
    "MITM-ArpSpoofing.pcap_Flow.csv"
]

# Find actual paths using filenames
file_map = {}

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename in sample_names:
            file_map[filename] = os.path.join(root, filename)

print("ACTUAL LABEL INSPECTION")
print("=" * 70)

for filename in sample_names:

    path = file_map.get(filename)

    if path is None:
        print(f"\nNOT FOUND: {filename}")
        continue

    print(f"\nFILE: {filename}")
    print("FOLDER:", os.path.basename(os.path.dirname(path)))

    # Read the actual Label column
    df = pd.read_csv(path, usecols=["Label"], low_memory=False)

    print("Rows:", len(df))
    print("Unique labels:", df["Label"].nunique())

    print("\nLabel distribution:")
    print(df["Label"].value_counts(dropna=False).head(20))

    print("\nFirst 10 label values:")
    print(df["Label"].head(10).tolist())

print("\nINSPECTION COMPLETED")

ACTUAL LABEL INSPECTION

FILE: BenignTraffic.pcap_Flow.csv
FOLDER: Benign
Rows: 183630
Unique labels: 1

Label distribution:
Label
NeedManualLabel    183630
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel']

FILE: DictionaryBruteForce.pcap_Flow.csv
FOLDER: Brute Force
Rows: 3619
Unique labels: 1

Label distribution:
Label
NeedManualLabel    3619
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel']

FILE: DDoS-ICMP_Flood.pcap_Flow.csv
FOLDER: DDOS
Rows: 10415
Unique labels: 1

Label distribution:
Label
NeedManualLabel    10415
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManualLabel', 

In [11]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

folder_files = {}

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            folder = os.path.basename(root)
            folder_files.setdefault(folder, []).append(filename)

print("DIAD FOLDER AUDIT")
print("=" * 70)

for folder, files in sorted(folder_files.items()):
    print(f"\nFolder: {folder}")
    print("CSV files:", len(files))
    print("Examples:")
    for filename in files[:5]:
        print("  ", filename)

print("\nTOTAL CSV FILES:", sum(len(files) for files in folder_files.values()))

# Search for documentation files
print("\nDOCUMENTATION FILE SEARCH")
print("=" * 70)

documentation = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower() in [
            "readme.txt", "readme.md", "readme",
            "description.txt", "metadata.txt"
        ]:
            documentation.append(os.path.join(root, filename))

if documentation:
    for path in documentation:
        print(path)
else:
    print("No README or metadata files found.")

DIAD FOLDER AUDIT

Folder: Benign
CSV files: 4
Examples:
   BenignTraffic.pcap_Flow.csv
   BenignTraffic1.pcap_Flow.csv
   BenignTraffic2.pcap_Flow.csv
   BenignTraffic3.pcap_Flow.csv

Folder: Brute Force
CSV files: 1
Examples:
   DictionaryBruteForce.pcap_Flow.csv

Folder: DDOS
CSV files: 61
Examples:
   DDoS-ACK_Fragmentation.pcap_Flow.csv
   DDoS-ACK_Fragmentation1.pcap_Flow.csv
   DDoS-ACK_Fragmentation10.pcap_Flow.csv
   DDoS-ACK_Fragmentation11.pcap_Flow.csv
   DDoS-ACK_Fragmentation12.pcap_Flow.csv

Folder: DOS
CSV files: 27
Examples:
   DoS-HTTP_Flood.pcap_Flow.csv
   DoS-HTTP_Flood1.pcap_Flow.csv
   DoS-SYN_Flood1.pcap_Flow.csv
   DoS-SYN_Flood3.pcap_Flow.csv
   DoS-SYN_Flood5.pcap_Flow.csv

Folder: Mirai
CSV files: 29
Examples:
   Mirai-greeth_flood.pcap_Flow.csv
   Mirai-greeth_flood1.pcap_Flow.csv
   Mirai-greeth_flood10.pcap_Flow.csv
   Mirai-greeth_flood11.pcap_Flow.csv
   Mirai-greeth_flood12.pcap_Flow.csv

Folder: Recon
CSV files: 1
Examples:
   VulnerabilityScan.pcap_F

In [12]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"
OUTPUT_DIR = "/kaggle/working/O1_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

records = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):

            folder = os.path.basename(root)

            records.append({
                "file": filename,
                "folder": folder,
                "proposed_label": folder,
                "full_path": os.path.join(root, filename)
            })

audit_df = pd.DataFrame(records)

print("PROPOSED LABEL AUDIT")
print("=" * 70)

print("\nFiles by proposed class:")
print(audit_df["proposed_label"].value_counts().sort_index())

print("\nTotal files:", len(audit_df))

print("\nMissing expected attack categories:")

expected = {
    "Benign", "Brute Force", "DDOS", "DOS",
    "Mirai", "Recon", "Spoofing", "Web-based"
}

observed = set(audit_df["folder"].unique())

for category in sorted(expected - observed):
    print("-", category)

print("\nUnexpected folders:")
for category in sorted(observed - expected):
    print("-", category)

output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_proposed_folder_label_audit.csv"
)

audit_df.to_csv(output_path, index=False)

print("\nAudit saved to:", output_path)
print("\nIMPORTANT: These are proposed folder-derived labels, not yet verified ground truth.")

PROPOSED LABEL AUDIT

Files by proposed class:
proposed_label
Benign          4
Brute Force     1
DDOS           61
DOS            27
Mirai          29
Recon           1
Spoofing        3
Name: count, dtype: int64

Total files: 126

Missing expected attack categories:
- Web-based

Unexpected folders:

Audit saved to: /kaggle/working/O1_results/O1_DIAD_proposed_folder_label_audit.csv

IMPORTANT: These are proposed folder-derived labels, not yet verified ground truth.


In [13]:
import pandas as pd
import os

report_path = (
    "/kaggle/working/O1_results/"
    "O1_DIAD_corrected_file_quality.csv"
)

quality_df = pd.read_csv(report_path)

print("QUALITY REPORT COLUMNS:")
print(quality_df.columns.tolist())

print("\nFIRST 10 ROWS:")
display(quality_df.head(10))

print("\nREPORT DIMENSIONS:")
print(quality_df.shape)

QUALITY REPORT COLUMNS:
['File', 'Rows', 'Missing', 'Infinite', 'Within_Chunk_Duplicates', 'Status']

FIRST 10 ROWS:


,File,Rows,Missing,Infinite,Within_Chunk_Duplicates,Status
0,BenignTraffic.pcap_Flow.csv,183630,10,60,7,Success
1,BenignTraffic1.pcap_Flow.csv,84526,7,63,4,Success
2,BenignTraffic2.pcap_Flow.csv,91279,14,76,4,Success
3,BenignTraffic3.pcap_Flow.csv,38895,1,33,1,Success
4,DictionaryBruteForce.pcap_Flow.csv,3619,0,0,0,Success
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,323886,5,15,10,Success
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,137172,2,10,5,Success
7,DDoS-ACK_Fragmentation10.pcap_Flow.csv,166796,7,11,4,Success
8,DDoS-ACK_Fragmentation11.pcap_Flow.csv,198622,1,5,1,Success
9,DDoS-ACK_Fragmentation12.pcap_Flow.csv,37292,2,10,3,Success



REPORT DIMENSIONS:
(126, 6)


In [14]:
import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

quality_path = os.path.join(
    OUTPUT_DIR, "O1_DIAD_corrected_file_quality.csv"
)

audit_path = os.path.join(
    OUTPUT_DIR, "O1_DIAD_proposed_folder_label_audit.csv"
)

quality_df = pd.read_csv(quality_path)
audit_df = pd.read_csv(audit_path)

# Normalize filenames for matching
def normalize_filename(value):
    return os.path.basename(str(value)).replace("\\", "").strip()

quality_df["File_key"] = quality_df["File"].apply(normalize_filename)
audit_df["File_key"] = audit_df["file"].apply(normalize_filename)

# Attach proposed folder labels to quality measurements
merged = quality_df.merge(
    audit_df[["File_key", "folder", "proposed_label"]],
    on="File_key",
    how="left",
    validate="one_to_one"
)

print("FILE MATCHING CHECK")
print("=" * 65)
print("Quality report files:", len(quality_df))
print("Matched files:", merged["proposed_label"].notna().sum())
print("Unmatched files:", merged["proposed_label"].isna().sum())

if merged["proposed_label"].isna().any():
    print("\nUnmatched filenames:")
    print(merged.loc[
        merged["proposed_label"].isna(), "File"
    ].tolist())

# Aggregate by proposed class
class_summary = (
    merged.groupby("proposed_label", dropna=False)
    .agg(
        Number_of_files=("File", "count"),
        Total_records=("Rows", "sum"),
        Missing_values=("Missing", "sum"),
        Infinite_values=("Infinite", "sum"),
        Within_chunk_duplicates=("Within_Chunk_Duplicates", "sum"),
        Successful_files=("Status", lambda x: (x == "Success").sum())
    )
    .reset_index()
)

class_summary["Missing_rate_percent"] = (
    class_summary["Missing_values"]
    / class_summary["Total_records"] * 100
).round(4)

class_summary["Infinite_rate_per_1000_records"] = (
    class_summary["Infinite_values"]
    / class_summary["Total_records"] * 1000
).round(4)

class_summary = class_summary.sort_values(
    "Total_records", ascending=False
)

print("\nPROPOSED CLASS-LEVEL DATA QUALITY SUMMARY")
print("=" * 65)
display(class_summary)

# Save report
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_proposed_class_quality_summary.csv"
)

class_summary.to_csv(output_path, index=False)

print("\nReport saved to:", output_path)

print("\nIMPORTANT:")
print("Class labels are provisional folder-derived labels.")
print("Duplicate counts are within-chunk counts, not global duplicates.")

FILE MATCHING CHECK
Quality report files: 126
Matched files: 126
Unmatched files: 0

PROPOSED CLASS-LEVEL DATA QUALITY SUMMARY


,proposed_label,Number_of_files,Total_records,Missing_values,Infinite_values,Within_chunk_duplicates,Successful_files,Missing_rate_percent,Infinite_rate_per_1000_records
3,DOS,27,14853092,83129,87259,27213,27,0.5597,5.8748
2,DDOS,61,3478814,1711,2131,5407,61,0.0492,0.6126
5,Recon,1,442158,521,1733,22,1,0.1178,3.9194
0,Benign,4,398330,32,232,16,4,0.0080,0.5824
4,Mirai,29,174588,38,102,201,29,0.0218,0.5842
6,Spoofing,3,157238,76,132,71,3,0.0483,0.8395
1,Brute Force,1,3619,0,0,0,1,0.0000,0.0000



Report saved to: /kaggle/working/O1_results/O1_DIAD_proposed_class_quality_summary.csv

IMPORTANT:
Class labels are provisional folder-derived labels.
Duplicate counts are within-chunk counts, not global duplicates.


In [15]:
import os
import re
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

records = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):

            folder = os.path.basename(root)

            # Remove CSV and flow suffix
            scenario = filename.replace(".csv", "")
            scenario = scenario.replace(".pcap_Flow", "")

            # Remove trailing capture numbers
            scenario = re.sub(r"\d+$", "", scenario)

            records.append({
                "Filename": filename,
                "Folder": folder,
                "Scenario": scenario
            })

validation_df = pd.DataFrame(records)

print("FOLDER AND SCENARIO VALIDATION")
print("=" * 70)

print("Total files:", len(validation_df))

print("\nFiles per folder:")
print(validation_df["Folder"].value_counts())

print("\nUnique scenarios per folder:")

for folder, group in validation_df.groupby("Folder"):
    print(f"\n{folder}:")
    for scenario in sorted(group["Scenario"].unique()):
        print("  ", scenario)

output_path = "/kaggle/working/O1_results/O1_DIAD_scenario_validation.csv"

validation_df.to_csv(output_path, index=False)

print("\nValidation report saved:", output_path)

FOLDER AND SCENARIO VALIDATION
Total files: 126

Files per folder:
Folder
DDOS           61
Mirai          29
DOS            27
Benign          4
Spoofing        3
Brute Force     1
Recon           1
Name: count, dtype: int64

Unique scenarios per folder:

Benign:
   BenignTraffic

Brute Force:
   DictionaryBruteForce

DDOS:
   DDoS-ACK_Fragmentation
   DDoS-HTTP_Flood-
   DDoS-ICMP_Flood
   DDoS-ICMP_Fragmentation

DOS:
   DoS-HTTP_Flood
   DoS-SYN_Flood
   DoS-TCP_Flood
   DoS-UDP_Flood

Mirai:
   Mirai-greeth_flood

Recon:
   VulnerabilityScan

Spoofing:
   DNS_Spoofing
   MITM-ArpSpoofing

Validation report saved: /kaggle/working/O1_results/O1_DIAD_scenario_validation.csv


In [16]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

# Load the existing file-level scenario audit
input_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_validation.csv"
)

df = pd.read_csv(input_path)

# Explicit folder-to-label mapping
label_mapping = {
    "Benign": "Benign",
    "Brute Force": "Brute Force",
    "DDOS": "DDoS",
    "DOS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing"
}

# Create a separate proposed label column
df["Derived_Label"] = df["Folder"].map(label_mapping)

# Validate mapping
print("=" * 65)
print("DIAD DERIVED-LABEL MAPPING VALIDATION")
print("=" * 65)

print("Total files:", len(df))
print("Successfully mapped:", df["Derived_Label"].notna().sum())
print("Unmapped files:", df["Derived_Label"].isna().sum())

if df["Derived_Label"].isna().any():
    print("\nUnmapped folders:")
    print(df.loc[df["Derived_Label"].isna(), "Folder"].unique())
    raise ValueError("Some folders have no label mapping.")

# Show class-level file counts
print("\nFILES PER DERIVED CLASS:")
print(df["Derived_Label"].value_counts())

# Show scenario-to-label mapping
print("\nSCENARIO AND DERIVED LABEL:")
display(
    df[["Filename", "Folder", "Scenario", "Derived_Label"]].head(15)
)

# Save mapping report
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_file_derived_label_mapping.csv"
)

df.to_csv(output_path, index=False)

print("\nMapping report saved:")
print(output_path)

print("\nNOTE: Derived labels are provisional and folder-based.")
print("Original CSV Label values have not been modified.")


DIAD DERIVED-LABEL MAPPING VALIDATION
Total files: 126
Successfully mapped: 126
Unmapped files: 0

FILES PER DERIVED CLASS:
Derived_Label
DDoS           61
Mirai          29
DoS            27
Benign          4
Spoofing        3
Brute Force     1
Recon           1
Name: count, dtype: int64

SCENARIO AND DERIVED LABEL:


,Filename,Folder,Scenario,Derived_Label
0,BenignTraffic.pcap_Flow.csv,Benign,BenignTraffic,Benign
1,BenignTraffic1.pcap_Flow.csv,Benign,BenignTraffic,Benign
2,BenignTraffic2.pcap_Flow.csv,Benign,BenignTraffic,Benign
3,BenignTraffic3.pcap_Flow.csv,Benign,BenignTraffic,Benign
4,DictionaryBruteForce.pcap_Flow.csv,Brute Force,DictionaryBruteForce,Brute Force
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,DDOS,DDoS-ACK_Fragmentation,DDoS
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,DDOS,DDoS-ACK_Fragmentation,DDoS
7,DDoS-ACK_Fragmentation10.pcap_Flow.csv,DDOS,DDoS-ACK_Fragmentation,DDoS
8,DDoS-ACK_Fragmentation11.pcap_Flow.csv,DDOS,DDoS-ACK_Fragmentation,DDoS
9,DDoS-ACK_Fragmentation12.pcap_Flow.csv,DDOS,DDoS-ACK_Fragmentation,DDoS



Mapping report saved:
/kaggle/working/O1_results/O1_DIAD_file_derived_label_mapping.csv

NOTE: Derived labels are provisional and folder-based.
Original CSV Label values have not been modified.


In [17]:

import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "/kaggle/working/O1_results"

quality_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_corrected_file_quality.csv"
)

mapping_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_file_derived_label_mapping.csv"
)

quality = pd.read_csv(quality_path)
mapping = pd.read_csv(mapping_path)

# Match filenames
def normalize_filename(value):
    return os.path.basename(str(value)).replace("\\", "").strip()

quality["File_key"] = quality["File"].apply(normalize_filename)
mapping["File_key"] = mapping["Filename"].apply(normalize_filename)

# Merge record counts with proposed labels
verified = quality.merge(
    mapping[["File_key", "Derived_Label"]],
    on="File_key",
    how="left",
    validate="one_to_one"
)

# Check matching
print("=" * 65)
print("DERIVED-LABEL RECORD COUNT VERIFICATION")
print("=" * 65)

print("Quality report files:", len(quality))
print("Mapping report files:", len(mapping))
print("Matched files:", verified["Derived_Label"].notna().sum())
print("Unmatched files:", verified["Derived_Label"].isna().sum())

if verified["Derived_Label"].isna().any():
    raise ValueError("Unmatched files detected. Stop verification.")

# Calculate record distribution
summary = (
    verified.groupby("Derived_Label")
    .agg(
        Number_of_files=("File", "count"),
        Records=("Rows", "sum")
    )
    .reset_index()
)

total_records = summary["Records"].sum()

summary["Percentage"] = (
    summary["Records"] / total_records * 100
).round(4)

summary = summary.sort_values(
    "Records", ascending=False
)

print("\nRECORD DISTRIBUTION:")
display(summary)

print("\nTOTAL RECORDS:", total_records)

# Calculate proposed imbalance ratio
majority = summary["Records"].max()
minority = summary["Records"].min()

print("Largest class:", majority)
print("Smallest class:", minority)
print("Proposed imbalance ratio:", round(majority / minority, 2))

# Save verification report
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_derived_label_record_verification.csv"
)

summary.to_csv(output_path, index=False)

print("\nVerification report saved:")
print(output_path)

print("\nNOTE: These statistics use provisional folder-derived labels.")


DERIVED-LABEL RECORD COUNT VERIFICATION
Quality report files: 126
Mapping report files: 126
Matched files: 126
Unmatched files: 0

RECORD DISTRIBUTION:


,Derived_Label,Number_of_files,Records,Percentage
3,DoS,27,14853092,76.1391
2,DDoS,61,3478814,17.8329
5,Recon,1,442158,2.2666
0,Benign,4,398330,2.0419
4,Mirai,29,174588,0.8950
6,Spoofing,3,157238,0.8060
1,Brute Force,1,3619,0.0186



TOTAL RECORDS: 19507839
Largest class: 14853092
Smallest class: 3619
Proposed imbalance ratio: 4104.2

Verification report saved:
/kaggle/working/O1_results/O1_DIAD_derived_label_record_verification.csv

NOTE: These statistics use provisional folder-derived labels.


In [18]:
import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/aymeneinformatique/cic-diad-new-2024"

print("Checking existing DIAD dataset...")

for root, dirs, files in os.walk(DATASET_PATH):
    csv_files = [f for f in files if f.endswith(".csv")]

    for filename in csv_files[:3]:
        path = os.path.join(root, filename)

        print("\nFile:", filename)

        df = pd.read_csv(path, nrows=5)

        print("Columns:")
        print(df.columns.tolist())

        if "Label" in df.columns:
            print("\nOriginal Label values:")
            print(df["Label"].value_counts(dropna=False))

        elif "Label 2 for AD" in df.columns:
            print("\nOriginal AD labels:")
            print(df["Label 2 for AD"].value_counts(dropna=False))

Checking existing DIAD dataset...

File: BenignTraffic.pcap_Flow.csv
Columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag C